# Cleaning the ethnicity dataset

This notebook inspects and cleans the official Latvian migration dataset `ibe040-migration-by-ethnicity-2016-2025-raw.csv`.
The goal is to inspect the raw data, convert the wide format year columns into a long format dataset, clean the dataset, validate the result after melting and cleaning, and save the dataset in `data/interim` folder for subsequent transformation.

In [1]:
import pandas as pd

ethnicity_data = pd.read_csv("../data/raw/ibe040-migration-by-ethnicity-2016-2025-raw.csv", skiprows=2)

ethnicity_data.head()

,Tautība,Rādītāji,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
0,Pavisam,Iebrauca Latvijā,8345,9916,10909,11223,8840,12689,38708,15629,14701,13826
1,Pavisam,Izbrauca no Latvijas,20574,17724,15814,14583,11990,12975,16680,18752,18937,15117
2,Latvieši,Iebrauca Latvijā,3081,3039,2888,3078,2849,4069,5214,5827,5399,4295
3,Latvieši,Izbrauca no Latvijas,10379,9381,7538,6073,4820,5389,6584,5424,5354,4839
4,Afgāņi,Iebrauca Latvijā,…,…,…,…,…,…,…,11,.,11


The required libraries are imported and the raw country groups dataset is loaded.
The first two rows of the `.csv` file contain the dataset title and a blank line before the actual column names, therefore, `skiprows=2` is used so that pandas reads the correct header row.
The first rows confirm that the dataset was loaded with the expected identifier columns, `Tautība` and `Rādītāji`, and the year columns from 2016 to 2025.

In [2]:
ethnicity_data.shape

(96, 12)

The dataset contains 96 rows and 12 columns.
The rows represent two migration indicators for 48 ethnicity categories. The columns consist of two identifier columns, `Tautība` and `Rādītāji`, and ten year columns from 2016 to 2025.
The shape, together with the displayed first rows, confirms that the dataset is structured in a wide format. Additionally, the first rows show that some migration counts are recorded as `…` and `.`. This means that the data are missing for one or another reason, and must be handled later in the cleaning process.

In [3]:
ethnicity_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 96 entries, 0 to 95
Data columns (total 12 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   Tautība   96 non-null     str  
 1   Rādītāji  96 non-null     str  
 2   2016      96 non-null     str  
 3   2017      96 non-null     str  
 4   2018      96 non-null     str  
 5   2019      96 non-null     str  
 6   2020      96 non-null     str  
 7   2021      96 non-null     str  
 8   2022      96 non-null     str  
 9   2023      96 non-null     str  
 10  2024      96 non-null     str  
 11  2025      96 non-null     str  
dtypes: str(12)
memory usage: 9.1 KB


The dataset contains 96 non-null entries for every column.
`Tautība`, `Rādītāji`, and the ten year columns use pandas default `str` data type. All required columns in the dataset are present, but each year column contains at least one `…` or `.` placeholder, as discovered earlier. These placeholders keep the year columns as `str` and must be handled before the migration counts can be converted to a numeric data type.

In [4]:
identifier_columns = ["Tautība", "Rādītāji"]

for column in identifier_columns:
    print(ethnicity_data[column].value_counts(dropna=False))

Tautība
Pavisam                 2
Latvieši                2
Afgāņi                  2
Amerikāņi               2
Angļi                   2
Armēņi                  2
Azerbaidžāņi            2
Baltkrievi              2
Brazīlieši              2
Bulgāri                 2
Dāņi                    2
Ebreji                  2
Ēģiptieši               2
Francūži                2
Grieķi                  2
Gruzīni                 2
Igauņi                  2
Indieši                 2
Irāņi                   2
Itāļi                   2
Kanādieši               2
Kazahi                  2
Kirgīzi                 2
Korejieši               2
Krievi                  2
Ķīnieši                 2
Lietuvieši              2
Moldāvi                 2
Nepālieši               2
Norvēģi                 2
Poļi                    2
Portugāļi               2
Romi (čigāni)           2
Rumāņi                  2
Sīrieši                 2
Somi                    2
Spāņi                   2
Tadžiki                 2
Tatā

The identifier columns contain only the expected values:
- `Rādītāji` contains two migration identifiers: `Iebrauca Latvijā` and `Izbrauca no Latvijas`. Each appears 48 times, once for each ethnicity category.
- `Tautība` contains 48 ethnicity categories. Each appears twice: once for arrivals and once for departures.

In [5]:
print("Count of missing values:")
print(ethnicity_data.isna().sum())

print("Count of empty strings:")
print(ethnicity_data[identifier_columns].eq("").sum())

Count of missing values:
Tautība     0
Rādītāji    0
2016        0
2017        0
2018        0
2019        0
2020        0
2021        0
2022        0
2023        0
2024        0
2025        0
dtype: int64
Count of empty strings:
Tautība     0
Rādītāji    0
dtype: int64


The dataset does not contain missing values or empty strings in any column. However, the year columns contain `…` or `.` placeholders instead of numeric values, as discovered earlier.
This means that checking for missing values with `.isna()` and empty strings with `.eq("")` does not detect these placeholders. They will be handled accordingly later in the cleaning process.

In [6]:
for column in identifier_columns:
    stripped_values = ethnicity_data[column].astype("string").str.strip()

    whitespace_count = stripped_values.ne(ethnicity_data[column].astype("string")).sum()

    print(f"{column}: {whitespace_count} values with leading or trailing whitespace.")

Tautība: 0 values with leading or trailing whitespace.
Rādītāji: 0 values with leading or trailing whitespace.


The identifier columns are checked for leading or trailing whitespace, which may not be visible when the values are displayed.
For each identifier column, the values are stripped of leading and trailing whitespace and compared with the original values.
Bot identifier columns contain no leading or trailing whitespace, therefore, no whitespace cleaning is required.

In [7]:
year_columns = ethnicity_data.columns[2:]

for column in year_columns:
    ellipsis_count = (ethnicity_data[column] == "…").sum()
    dot_count = (ethnicity_data[column] == ".").sum()

    print(f"{column}: … = {ellipsis_count}, . = {dot_count}")

2016: … = 38, . = 0
2017: … = 30, . = 0
2018: … = 32, . = 0
2019: … = 24, . = 0
2020: … = 26, . = 0
2021: … = 30, . = 0
2022: … = 16, . = 0
2023: … = 3, . = 22
2024: … = 4, . = 23
2025: … = 3, . = 24


The year columns contain `…` and `.` placeholders.
According to the dataset metadata, `…` indicates that data are not available or too uncertain for presentation. `.` indicates that data were not released for confidentiality reasons.
The `…` placeholder appears in every year, while `.` appears from 2023 onwards. Both must be treated as unavailable migration counts and converted to missing values after melting the dataset.

In [8]:
ethnicity_data[identifier_columns].duplicated().sum()

np.int64(0)

The identifier columns are checked for duplicate entries in the dataset.
Result of `0` confirms that no duplicates are present. No duplicate rows need to be removed.

In [9]:
ethnicity_melted = ethnicity_data.melt(
    id_vars=identifier_columns,
    value_vars=year_columns,
    var_name="Gads",
    value_name="Skaits"
)

ethnicity_melted.head()

,Tautība,Rādītāji,Gads,Skaits
0,Pavisam,Iebrauca Latvijā,2016,8345
1,Pavisam,Izbrauca no Latvijas,2016,20574
2,Latvieši,Iebrauca Latvijā,2016,3081
3,Latvieši,Izbrauca no Latvijas,2016,10379
4,Afgāņi,Iebrauca Latvijā,2016,…


The dataset is converted from wide format to long format.
`year_columns` selects the ten year columns. The `.melt()` method keeps the identifier columns, and converts each year column into a row. The former year column names are stored in `Gads`, and their published migration counts are stored in `Skaits`.
The resulting dataset contains one row for every migration indicator, ethnicity category, and year.

In [10]:
ethnicity_melted["Skaits"] = ethnicity_melted["Skaits"].replace({
    "…": pd.NA,
    ".": pd.NA
})

ethnicity_melted.info()

<class 'pandas.DataFrame'>
RangeIndex: 960 entries, 0 to 959
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   Tautība   960 non-null    str  
 1   Rādītāji  960 non-null    str  
 2   Gads      960 non-null    str  
 3   Skaits    685 non-null    str  
dtypes: str(4)
memory usage: 30.1 KB


The placeholders discussed earlier are replaced with missing values.
After this replacement, the `Skaits` column has fewer non-null entries than the other columns. This indicates that 275 migration counts are now recorded as missing values (960 - 685 = 275).

In [11]:
ethnicity_melted["Gads"] = ethnicity_melted["Gads"].astype("int64")

ethnicity_melted["Skaits"] = pd.to_numeric(
    ethnicity_melted["Skaits"],
    errors="coerce"
)

After melting, `Gads` column contains former year column names. During melting, these column names are stored as strings, so `Gads` has the `str` data type.
`Gads` is converted to `int64`, matching the data type of the year columns in the original dataset. This keeps the years as numeric, ordered values for later transformations, such as filtering or comparing values by year, if necessary.
`Skaits` column is converted to a numeric data type using `pd.to_numeric()`. Because this column contains missing values, it is stored as `float64` after conversion. The missing values represent unavailable data. They are not treated as zero or replaced with an imputed value because zero would incorrectly indicate that no migration occurred, while an imputed value would create a migration count that was not actually observed.

In [12]:
ethnicity_melted.shape

(960, 4)

The melted dataset contains 960 rows and 4 columns.
The rows represent every combination of two migrations indicators, 48 ethnicity categories, and ten years.
The four columns now are `Tautība`, `Rādītāji`, `Gads`, and `Skaits`.
This confirms that the melting from wide to long format has been successful.

In [13]:
ethnicity_melted.info()

<class 'pandas.DataFrame'>
RangeIndex: 960 entries, 0 to 959
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Tautība   960 non-null    str    
 1   Rādītāji  960 non-null    str    
 2   Gads      960 non-null    int64  
 3   Skaits    685 non-null    float64
dtypes: float64(1), int64(1), str(2)
memory usage: 30.1 KB


`Tautība`, `Rādītāji` and `Gads` columns in the newly melted dataset contain no null values across its 960 rows.
`Tautība` and `Rādītāji` keep `str` data type as intended. `Gads` uses `int64` data type.
`Skaits`, as mentioned earlier, contains values that were converted to missing rather than imputed or left as placeholders, and it uses `float64` data type.
This confirms that the conversion of years and counts has been successful.

In [14]:
print("All available counts are non-negative:")
print((ethnicity_melted["Skaits"].dropna() >= 0).all())

print("All available counts are whole numbers:")
print((ethnicity_melted["Skaits"].dropna() % 1 == 0).all())

All available counts are non-negative:
True
All available counts are whole numbers:
True


The cleaned migration counts are checked to confirm that they are valid. `Skaits >= 0` checks that every available count is non-negative. `Skaits % 1 == 0` checks that every available count is a whole number.
Missing values are removed with `.dropna()` before these checks because they represent unavailable data rather than invalid migration counts. Including them would prevent the validity checks from working correctly.
Both checks return `True`, confirming that no invalid values were introduced by accident during cleaning.

In [15]:
checks_to_compare = [
    ("Iebrauca Latvijā", "Pavisam", 2016, 8345),
    ("Iebrauca Latvijā", "Pavisam", 2025, 13826),
    ("Izbrauca no Latvijas", "Pavisam", 2016, 20574),
    ("Izbrauca no Latvijas", "Pavisam", 2025, 15117)
]

for indicator, ethnicity, year, expected_count in checks_to_compare:
    actual_count = ethnicity_melted.loc[
        (ethnicity_melted["Rādītāji"] == indicator)
        & (ethnicity_melted["Tautība"] == ethnicity)
        & (ethnicity_melted["Gads"] == year),
        "Skaits"
    ].item()

    print(f"{indicator} | {ethnicity} | {year}: expected {expected_count}, found {actual_count}")

Iebrauca Latvijā | Pavisam | 2016: expected 8345, found 8345.0
Iebrauca Latvijā | Pavisam | 2025: expected 13826, found 13826.0
Izbrauca no Latvijas | Pavisam | 2016: expected 20574, found 20574.0
Izbrauca no Latvijas | Pavisam | 2025: expected 15117, found 15117.0


Selected rows are checked to confirm that they still match the correct migration flow, ethnicity, and year after melting.
For each check, the expected count is taken from the originally loaded dataset. The melted dataset is filtered by migration indicator, ethnicity, and year, and the resulting `Skaits` value is compared with the expected count.
All selected values match. The found values are displayed as floats, such as `8345.0`, because of the conversion of `Skaits` column. This does not affect the comparison, as each displayed value is still a whole number.
These checks confirm that the melting preserved the correct migration flows, ethnicities, years and published counts.

In [16]:
ethnicity_melted.to_csv("../data/interim/ibe040-migration-by-ethnicity-2016-2025-interim.csv", index=False, encoding="utf-8-sig")
print("The dataset has been saved")

The dataset has been saved


The cleaned long format dataset is saved to the `data/interim` folder for use in transformations as the next step in the planned methodology.
`index=False` prevents pandas from writing its numeric row index as an extra column in the `.csv` file. `encoding="utf-8-sig"` writes a UTF-8 file with a bite order mark, which helps some programs, including Microsoft Excel, to correctly recognize the encoding and display Latvian characters.